# Dimensionality Reduction: PCA vs t-SNE vs UMAP

This notebook compares three popular dimensionality reduction techniques for visualizing high-dimensional data.


In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import umap
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Load data
df = pd.read_csv('student_placement_prediction_dataset_2026.csv')
print(f"Dataset shape: {df.shape}")
print(f"\nFeatures: {df.columns.tolist()}")
df.head()

In [ ]:
# Select numeric features
features = ['cgpa', 'aptitude_score', 'coding_skill_score']
X = df[features].copy()
X = X.fillna(X.median()).fillna(0)

# Standardize features
X_scaled = StandardScaler().fit_transform(X)
print(f"Feature matrix shape: {X_scaled.shape}")
print(f"Features used: {features}")

## 1. PCA (Principal Component Analysis)

PCA is a linear dimensionality reduction method that finds orthogonal directions of maximum variance.


In [ ]:
# Apply PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print(f"PCA Output shape: {X_pca.shape}")
print(f"\nExplained Variance Ratio: {pca.explained_variance_ratio_}")
print(f"Total Variance Retained: {sum(pca.explained_variance_ratio_) * 100:.2f}%")

## 2. t-SNE (t-Distributed Stochastic Neighbor Embedding)

t-SNE is a non-linear method that preserves local structure and is excellent for visualization.


In [ ]:
# Apply t-SNE
tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
X_tsne = tsne.fit_transform(X_scaled)

print(f"t-SNE Output shape: {X_tsne.shape}")
print("t-SNE focuses on preserving local neighborhoods and is great for visualization.")

## 3. UMAP (Uniform Manifold Approximation and Projection)

UMAP is a non-linear method that preserves both local and global structure.


In [ ]:
# Apply UMAP
umap_model = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=42)
X_umap = umap_model.fit_transform(X_scaled)

print(f"UMAP Output shape: {X_umap.shape}")
print("UMAP preserves both local neighborhoods and global structure.")

## Visualization Comparison


In [ ]:
# Cluster data using K-Means (3 clusters)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
labels = kmeans.fit_predict(X_pca)  # Using PCA for consistency

# Create comparison plots
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# PCA
axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=labels, cmap='viridis', edgecolors='black', alpha=0.7)
axes[0].set_xlabel('Principal Component 1')
axes[0].set_ylabel('Principal Component 2')
axes[0].set_title('PCA Clustering')

# t-SNE
axes[1].scatter(X_tsne[:, 0], X_tsne[:, 1], c=labels, cmap='viridis', edgecolors='black', alpha=0.7)
axes[1].set_xlabel('t-SNE Dimension 1')
axes[1].set_ylabel('t-SNE Dimension 2')
axes[1].set_title('t-SNE Clustering')

# UMAP
axes[2].scatter(X_umap[:, 0], X_umap[:, 1], c=labels, cmap='viridis', edgecolors='black', alpha=0.7)
axes[2].set_xlabel('UMAP Dimension 1')
axes[2].set_ylabel('UMAP Dimension 2')
axes[2].set_title('UMAP Clustering')

plt.tight_layout()
plt.show()

print("All three methods are applied to the same data and colored by the same clusters!")

## Metrics Comparison


In [ ]:
# Calculate metrics for each method
methods = {
    'PCA': X_pca,
    't-SNE': X_tsne,
    'UMAP': X_umap
}

comparison_results = []

for method_name, X_reduced in methods.items():
    kmeans_temp = KMeans(n_clusters=3, random_state=42, n_init=10)
    temp_labels = kmeans_temp.fit_predict(X_reduced)
    
    inertia = kmeans_temp.inertia_
    silhouette = silhouette_score(X_reduced, temp_labels)
    
    comparison_results.append({
        'Method': method_name,
        'Inertia': round(inertia, 4),
        'Silhouette Score': round(silhouette, 4)
    })
    
    print(f"{method_name}:")
    print(f"  Inertia: {inertia:.4f}")
    print(f"  Silhouette Score: {silhouette:.4f}")
    print()

comparison_df = pd.DataFrame(comparison_results)
print("\nComparison Table:")
print(comparison_df.to_string(index=False))

## Key Differences

### PCA
- **Linear** dimensionality reduction
- Preserves **global variance** structure
- Fast computation
- Good for understanding major patterns
- Variance retained (%): High

### t-SNE
- **Non-linear** dimensionality reduction
- Preserves **local neighborhood** structure
- Slower computation (especially for large datasets)
- Excellent for visualization and cluster discovery
- Can reveal intricate cluster structures

### UMAP
- **Non-linear** dimensionality reduction
- Preserves **both local and global** structure
- Faster than t-SNE
- Better at preserving global topology
- Good balance between speed and visualization quality


In [ ]:
# Summary
print("\n" + "="*60)
print("SUMMARY")
print("="*60)
print("\nUse PCA when:")
print("  - You need fast computation")
print("  - You want to understand global variance structure")
print("  - Interpretability is important")
print("\nUse t-SNE when:")
print("  - You want detailed visualization of clusters")
print("  - Local structure preservation is critical")
print("  - Dataset size is manageable (< 100,000 samples)")
print("\nUse UMAP when:")
print("  - You need fast, non-linear reduction")
print("  - You want both local and global structure")
print("  - Working with large datasets")
print("  - You need to preserve manifold structure")